In [ ]:
# === ONE-CELL: Merge LoRA and serve with vLLM (+bitsandbytes) ===
# Works with vllm 0.10.1.1 (latest on PyPI, Aug 2025)

# 1) Install deps
!pip -q install "transformers>=4.55.0" "peft>=0.13.2" "safetensors>=0.4.3" \
                "huggingface_hub>=0.26.0" "bitsandbytes>=0.43.0" \
                "vllm==0.10.1.1"

import os, sys, json, subprocess, shlex, time
from pathlib import Path

# ----------------------- CONFIG -----------------------
BASE_REPO     = "openai/gpt-oss-20b"              # fp16/bf16 base
ADAPTER_REPO  = "jbaghiro/gpt-oss-20b-adapter"    # HF LoRA repo
ADAPTER_DIR   = ""                                # optional: local adapter path
OUT_DIR       = "./gpt-oss-20b-merged-fp16"
HF_TOKEN      = ""                                # set if repo is private
TRUST_CODE    = True
VLLM_PORT     = 8000
MAX_MODEL_LEN = 8192
GPU_UTIL      = 0.95
# ------------------------------------------------------

from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

def download_repo(repo_id: str) -> Path:
    return Path(snapshot_download(repo_id=repo_id, token=HF_TOKEN or None))

def ensure_adapter_ok(p: Path):
    names = {x.name for x in p.glob("*")}
    if not ("adapter_config.json" in names or any(n.startswith("adapter_config") for n in names)):
        raise RuntimeError("missing adapter_config.json")
    if not any(n.endswith(".safetensors") and "adapter" in n for n in names):
        raise RuntimeError("missing adapter_model*.safetensors")

def merge_lora(base_dir: Path, adapter_dir: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    tok = AutoTokenizer.from_pretrained(str(base_dir), use_fast=True, trust_remote_code=TRUST_CODE)
    base = AutoModelForCausalLM.from_pretrained(str(base_dir), torch_dtype="auto", trust_remote_code=TRUST_CODE)
    merged = PeftModel.from_pretrained(base, str(adapter_dir), trust_remote_code=TRUST_CODE).merge_and_unload()
    tok.save_pretrained(str(out_dir))
    merged.save_pretrained(str(out_dir), safe_serialization=True)

def start_vllm(model_dir: Path, port: int = 8000):
    cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", str(model_dir),
        "--quantization", "bitsandbytes",
        "--dtype", "auto",
        "--gpu-memory-utilization", str(GPU_UTIL),
        "--max-model-len", str(MAX_MODEL_LEN),
        "--trust-remote-code",
        "--port", str(port),
        "--host", "0.0.0.0",
    ]
    print(" ".join(shlex.quote(c) for c in cmd))
    subprocess.Popen(cmd, stdout=open("vllm_server.log","w"), stderr=subprocess.STDOUT)
    time.sleep(2)
    print(f"[vllm] endpoint: http://localhost:{port}/v1/chat/completions")

# 3) Download base + adapter
base_dir = download_repo(BASE_REPO)
adapter_dir = Path(ADAPTER_DIR) if ADAPTER_DIR else download_repo(ADAPTER_REPO)
ensure_adapter_ok(adapter_dir)

# 4) Merge adapter into base
merge_lora(base_dir, adapter_dir, Path(OUT_DIR))

# 5) Launch vLLM in background
start_vllm(Path(OUT_DIR), port=VLLM_PORT)

print("\nTry a test request:")
print(f"""curl http://localhost:{VLLM_PORT}/v1/chat/completions \\
  -s -H "Content-Type: application/json" \\
  -d '{{"model":"{OUT_DIR}","messages":[{{"role":"user","content":"Give me 3 bullets about GPT-OSS."}}],"max_tokens":200}}'""")



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

USAGE_POLICY:   0%|          | 0.00/200 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

trainer_state.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/63.7M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
MXFP4 quantization requires triton >= 3.4.0 and kernels installed, we will default to dequantizing the model to bf16


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/bin/python -m vllm.entrypoints.openai.api_server --model gpt-oss-20b-merged-fp16 --quantization bitsandbytes --dtype auto --gpu-memory-utilization 0.95 --max-model-len 8192 --trust-remote-code --port 8000 --host 0.0.0.0
[vllm] endpoint: http://localhost:8000/v1/chat/completions

Try a test request:
curl http://localhost:8000/v1/chat/completions \
  -s -H "Content-Type: application/json" \
  -d '{"model":"./gpt-oss-20b-merged-fp16","messages":[{"role":"user","content":"Give me 3 bullets about GPT-OSS."}],"max_tokens":200}'


In [ ]:
!du -sh ./gpt-oss-20b-merged-fp16

39G	./gpt-oss-20b-merged-fp16


In [ ]:
!pip install -q "huggingface_hub>=0.26.0" safetensors

from huggingface_hub import HfApi, HfFolder, create_repo, upload_folder

# ---------------- CONFIG ----------------
HF_TOKEN   = ""   # <-- paste your HF write token here
LOCAL_DIR  = "./gpt-oss-20b-merged-fp16"   # path to the merged folder
HF_REPO    = "jbaghiro/gpt-oss-20b-merged-fp16"  # target repo on Hugging Face
PRIVATE    = False   # True if you want the repo private
BRANCH     = "main"
# ----------------------------------------

# Save token
HfFolder.save_token(HF_TOKEN)
api = HfApi(token=HF_TOKEN)

# Create repo if needed
create_repo(repo_id=HF_REPO, private=PRIVATE, exist_ok=True, repo_type="model")

# Upload folder (no multi_commits, for compatibility)
print(f"[upload] Uploading {LOCAL_DIR} → {HF_REPO} (branch={BRANCH}) …")
upload_folder(
    repo_id=HF_REPO,
    folder_path=LOCAL_DIR,
    repo_type="model",
    commit_message="Upload merged fp16 model",
    revision=BRANCH,
    token=HF_TOKEN,
)

print(f"✅ Done. View it at: https://huggingface.co/{HF_REPO}")



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip
[upload] Uploading ./gpt-oss-20b-merged-fp16 → jbaghiro/gpt-oss-20b-merged-fp16 (branch=main) …
